# Clustering

Groups survey rows into clusters using k-means or HDBSCAN on selected numeric columns and adds a `Cluster#number` column. Shows cluster sizes, mean profiles, and a 2D scatter visualization.

In [ ]:
# ── Colab bootstrap (no-op on Binder / JupyterHub) ────────────────────────
import os, sys
if "google.colab" in sys.modules:
    if not os.path.isfile("/tmp/suave-nb/helpers/suave_utils.py"):
        import subprocess
        _r = subprocess.run(
            ["git", "clone", "--depth=1", "https://github.com/izaslavsky/suave-notebooks.git", "/tmp/suave-nb"],
            capture_output=True, text=True
        )
        if _r.returncode != 0:
            raise RuntimeError(f"Could not clone suave-notebooks:\n{_r.stderr}")
    sys.path.insert(0, "/tmp/suave-nb/helpers")


In [ ]:
# ── Colab only: mount Google Drive to load session credentials ─────────────
import sys
if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')


In [ ]:
# ── Load SuAVE parameters ──────────────────────────────────────────────────
import sys, os, requests as _req, urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
from urllib.parse import urlparse as _urlparse
sys.path.insert(0, '../../helpers')
import suave_utils as su
from IPython.display import display, Markdown
import pandas as pd

def printmd(string):
    display(Markdown(string))

if not su.ENV.colab:
    # Binder / JupyterHub: parameters load automatically
    _p = su.load_params()
    if _p is None:
        raise RuntimeError('No SuAVE params. Open via SuAVEDispatch.ipynb first.')
else:
    # Colab: load from Drive automatically; prompt only if Drive has no session
    _p = su.load_params(_silent=True)
    if _p is None:
        from IPython.display import display, HTML
        import getpass
        display(HTML('<p style="color:#e07000">No session found on Drive. '
                     'Enter credentials from SuAVEDispatch:</p>'))
        _token = getpass.getpass('SUAVE_TOKEN: ')
        _host  = input('SUAVE_HOST (e.g. george.tirebiter.org): ')
        _p = su.load_params(token=_token, host=_host)

user          = _p.get('user', '')
survey_url    = _p.get('surveyurl', '')
csv_file      = _p.get('csv', '')
dzc_file      = _p.get('dzc', '')
active_object = _p.get('activeobject', 'none')
_caps         = su.detect_capabilities(_p)
views         = ','.join(_caps.get('views', []))
view          = 'grid'

localdzc             = _caps.get('localdzc', '')
full_images          = _caps.get('full_images', '')
full_images_location = full_images

absolutePath = os.path.expanduser('~/temp_csvs/')
os.makedirs(absolutePath, exist_ok=True)
_origin = _urlparse(survey_url)
_csv_url = f"{_origin.scheme}://{_origin.netloc}/surveys/{csv_file}"
_r = _req.get(_csv_url, timeout=30, verify=False)
_r.raise_for_status()
with open(absolutePath + csv_file, 'wb') as _f:
    _f.write(_r.content)

su.show_params(_p)

In [ ]:
import sys
sys.path.insert(1, '../../helpers')
import panel_libs as panellibs
import suave_integration as suaveint

import ipywidgets as widgets
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

## 1. Load data and configure clustering

In [ ]:
df = panellibs.extract_data(absolutePath + csv_file)
print(f"Loaded {len(df)} rows")

num_cols = [c for c in df.columns if '#number' in c]
if not num_cols:
    raise ValueError('No #number columns found.')

col_selector = widgets.SelectMultiple(
    options=num_cols, value=num_cols[:min(5, len(num_cols))],
    description='Columns:', rows=min(10, len(num_cols)),
    layout=widgets.Layout(width='70%', height='200px')
)
algo_selector = widgets.RadioButtons(
    options=['k-means', 'HDBSCAN'],
    description='Algorithm:'
)
k_slider = widgets.IntSlider(
    value=4, min=2, max=20, description='k (k-means):',
    continuous_update=False
)
min_samples = widgets.IntSlider(
    value=5, min=2, max=50, description='min_samples (HDBSCAN):',
    continuous_update=False
)

def _toggle(change):
    k_slider.disabled       = change['new'] == 'HDBSCAN'
    min_samples.disabled    = change['new'] == 'k-means'

algo_selector.observe(_toggle, names='value')
min_samples.disabled = True

printmd("**Select numeric columns and clustering parameters:**")
display(col_selector, algo_selector, k_slider, min_samples)

## 2. Cluster

In [ ]:
selected = list(col_selector.value)
algo     = algo_selector.value

X = df[selected].apply(pd.to_numeric, errors='coerce')
missing = X.isnull().sum().sum()
if missing > 0:
    printmd(f"**{missing} missing values imputed with column mean.**")
    X = X.fillna(X.mean())

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

if algo == 'k-means':
    k = k_slider.value
    model = KMeans(n_clusters=k, random_state=42, n_init='auto')
    labels = model.fit_predict(X_scaled)
    printmd(f"**k-means with k={k} completed.**")
else:
    try:
        import hdbscan
    except ImportError:
        print('Installing hdbscan...')
        import subprocess, sys as _sys
        subprocess.check_call([_sys.executable, '-m', 'pip', 'install', '-q', 'hdbscan'])
        import hdbscan
    clusterer = hdbscan.HDBSCAN(min_samples=min_samples.value)
    labels = clusterer.fit_predict(X_scaled)
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    n_noise    = (labels == -1).sum()
    printmd(f"**HDBSCAN found {n_clusters} clusters; {n_noise} noise points (labeled -1).**")

df['Cluster#number'] = labels

# Cluster size table
printmd('**Cluster sizes:**')
display(df['Cluster#number'].value_counts().sort_index().rename('count').to_frame())

# Mean profile
short_names = {c: c.split('#')[0] for c in selected}
profile_df = df[selected + ['Cluster#number']].copy()
profile_df = profile_df.rename(columns=short_names)
profile_df = profile_df.groupby('Cluster#number').mean().round(3)
printmd('**Mean feature values per cluster:**')
display(profile_df)

## 3. 2D scatter visualization

In [ ]:
# Project to 2D with PCA for visualization
pca2 = PCA(n_components=2)
coords = pca2.fit_transform(X_scaled)

unique_labels = sorted(set(labels))
cmap = plt.cm.get_cmap('tab20', len(unique_labels))

plt.figure(figsize=(8, 6))
for i, lbl in enumerate(unique_labels):
    mask = labels == lbl
    color = 'gray' if lbl == -1 else cmap(i)
    plt.scatter(coords[mask, 0], coords[mask, 1], s=15, alpha=0.6,
                color=color, label=f'Cluster {lbl}' if lbl != -1 else 'Noise')

plt.title('Cluster assignments (PCA 2D projection)')
plt.xlabel(f'PC1 ({pca2.explained_variance_ratio_[0]*100:.1f}%)')
plt.ylabel(f'PC2 ({pca2.explained_variance_ratio_[1]*100:.1f}%)')
plt.legend(markerscale=2, bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

## 4. Save and publish to SuAVE

In [ ]:
new_file = suaveint.save_csv_file(df, absolutePath, csv_file)

In [ ]:
#Input survey name
survey_name = input('Enter survey name: ')
printmd('**Survey Name:** ' + survey_name)
